# Phase 2 — Churn Prediction Prototype

Predict the **chance** (probability) that a customer churns, using the public
**IBM Telco Customer Churn** dataset (7,043 real telecom customers, ~26.5% churn).

Because the product is a *probability*, not just a yes/no label, this notebook treats
probability quality as first-class: **ROC-AUC** as the primary metric, a **calibration**
step so "70% risk" really means ~70%, and a **decision threshold chosen from business
costs** instead of a silent 0.5.

Pipeline shape (mirrors Phase 1): Acquire → Load & explore → Clean → Encode → Split →
Train models → Evaluate → Compare → Diagnose (calibrate + threshold + drivers) → Tune →
Save & predict.

## Step 1 — Acquire the dataset

Download the IBM Telco churn CSV (public sample data) once; skip if already present.

In [ ]:
from pathlib import Path
import urllib.request

DATA_PATH = Path("../data/telco_churn.csv")
URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("downloaded", DATA_PATH)
else:
    print("already present:", DATA_PATH)

## Step 2 — Load & explore

Look before modelling. Key questions: how big, how imbalanced, which columns are
categorical vs numeric, and is anything dirty?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head(3)

In [ ]:
print(df.dtypes)
print("\nchurn rate:", (df["Churn"] == "Yes").mean().round(4))
sns.countplot(data=df, x="Churn")
plt.title("Class balance — churn is the minority (~26.5%)")
plt.show()

In [ ]:
# Churn rate by contract type — a first hint at what drives churn
churn_by_contract = df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean())
print(churn_by_contract.round(3))

sns.histplot(data=df, x="tenure", hue="Churn", bins=30, multiple="stack")
plt.title("Churners cluster at low tenure")
plt.show()

In [ ]:
# Data-quality check: TotalCharges is typed as text (object) — why?
blanks = df["TotalCharges"].str.strip() == ""
print("blank TotalCharges rows:", blanks.sum())
print("their tenure values:", df.loc[blanks, "tenure"].unique())
# 11 brand-new customers (tenure=0) have a blank instead of 0 — classic real-data dirt.

**Findings:** 7,043 × 21; churn 26.5% (imbalanced — accuracy alone would mislead);
month-to-month contracts churn ~43% vs ~3% for two-year; churners concentrate at low
tenure. One real data bug: `TotalCharges` is text because 11 new customers (tenure=0)
have blank values.

## Step 3 — Clean

Fix exactly what Step 2 found, nothing speculative: coerce `TotalCharges` to numeric
(blanks → 0, justified because those customers have `tenure=0` so they've paid nothing),
drop the ID column, and encode the target as 0/1.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)

y = (df["Churn"] == "Yes").astype(int)          # 1 = churned
X = df.drop(columns=["customerID", "Churn"])     # customerID carries no signal

print("features:", X.shape, "| target churn rate:", y.mean().round(4))

## Step 4 — Encode features

Tabular data this time (Phase 1 was text). Categorical columns get one-hot encoding,
numeric columns get standardized. Both live inside a `ColumnTransformer` inside the
model `Pipeline`, so encoders are **fit on the training split only** — the same
no-data-leakage principle as Phase 1's TF-IDF.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL = [c for c in X.columns if c not in NUMERIC]

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])

# Demo only (real fit happens inside each model's Pipeline on the train split):
demo = make_preprocessor().fit_transform(X)
print("encoded matrix:", demo.shape)  # 19 raw columns -> ~45 model features

## Step 5 — Train / test split

Stratified 80/20 so the 26.5% churn ratio is preserved in both halves. The test set is
~1,409 customers — big enough that a single split is a trustworthy ruler (contrast with
Phase 1's 38-row split, which needed cross-validation to be honest).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"train={len(X_train)}  test={len(X_test)}  churn rate in test: {y_test.mean():.3f}")

## Step 6 — Train several models

Same swappable-pipeline discipline as Phase 1: every model is
`Pipeline([preprocess, classifier])`, trained identically, so the comparison is
apples-to-apples. XGBoost is optional (needs the libomp system library on macOS) —
the notebook degrades gracefully if it can't load.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

try:  # optional extra model
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric="logloss", random_state=42,
    )
except Exception as e:
    print("xgboost unavailable, skipping:", type(e).__name__)

fitted = {}
for name, clf in models.items():
    pipe = Pipeline([("pre", make_preprocessor()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    print("trained:", name)

## Step 7 — Evaluate (probability-first)

The deliverable is a churn *probability*, so we rank models by **ROC-AUC** (quality of
the probability ordering) and also report **PR-AUC** (more informative under imbalance)
and the **Brier score** (probability accuracy — lower is better). Label metrics at the
conventional 0.5 threshold are shown for reference only; the real threshold is chosen
in Step 9.

In [ ]:
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score,
                             precision_score, recall_score, roc_auc_score)

def evaluate(pipe):
    proba = pipe.predict_proba(X_test)[:, 1]
    label = (proba >= 0.5).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Brier": brier_score_loss(y_test, proba),
        "F1@0.5": f1_score(y_test, label),
        "Precision@0.5": precision_score(y_test, label),
        "Recall@0.5": recall_score(y_test, label),
    }

results = {name: evaluate(pipe) for name, pipe in fitted.items()}

## Step 8 — Compare

In [ ]:
comparison = (pd.DataFrame(results).T
              .sort_values("ROC-AUC", ascending=False)
              .round(3))
comparison

**Reading the table:** Logistic Regression leads on ROC-AUC (ranking quality), but
note its **Brier score is the worst** — `class_weight="balanced"` deliberately distorts
probabilities toward the minority class. Great ranking, dishonest percentages. That is
exactly what Step 9 fixes: in production, "this customer is 70% likely to churn" must
be a statement you can stand behind.

## Step 9 — Diagnose: calibrate, choose the threshold, explain the drivers

Three production concerns, in order:
1. **Calibration** — wrap the leader in `CalibratedClassifierCV` so predicted
   probabilities match observed frequencies.
2. **Threshold as a business decision** — a retention offer is cheap (~\$50), a lost
   customer is expensive (~\$500), so we sweep thresholds and pick the cheapest, which
   lands well below 0.5.
3. **Explainability** — which features drive churn (coefficients).

In [ ]:
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

leader = fitted["Logistic Regression"]

calibrated = CalibratedClassifierCV(
    Pipeline([("pre", make_preprocessor()),
              ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    method="sigmoid", cv=5,
)
calibrated.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_estimator(leader, X_test, y_test, n_bins=10,
                                  name="raw (balanced LogReg)", ax=ax)
CalibrationDisplay.from_estimator(calibrated, X_test, y_test, n_bins=10,
                                  name="calibrated (sigmoid)", ax=ax)
ax.set_title("Reliability curve — closer to the diagonal is better")
plt.show()

proba_raw = leader.predict_proba(X_test)[:, 1]
proba_cal = calibrated.predict_proba(X_test)[:, 1]
print(f"Brier raw={brier_score_loss(y_test, proba_raw):.3f}  "
      f"calibrated={brier_score_loss(y_test, proba_cal):.3f}")
print(f"ROC-AUC raw={roc_auc_score(y_test, proba_raw):.3f}  "
      f"calibrated={roc_auc_score(y_test, proba_cal):.3f}  (ranking preserved)")

In [ ]:
# Threshold sweep on CALIBRATED probabilities with a simple cost model:
# contacting a predicted churner costs $50 (retention offer);
# missing a real churner costs $500 (lost customer).
import numpy as np

OFFER_COST, CHURN_COST = 50, 500
rows = []
for th in np.arange(0.10, 0.75, 0.05):
    pred = (proba_cal >= th).astype(int)
    tp = int(((pred == 1) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    fn = int(((pred == 0) & (y_test == 1)).sum())
    rows.append({"threshold": round(th, 2),
                 "precision": precision_score(y_test, pred),
                 "recall": recall_score(y_test, pred),
                 "cost_$": OFFER_COST * (tp + fp) + CHURN_COST * fn})
sweep = pd.DataFrame(rows).round(3)
CHURN_THRESHOLD = float(sweep.loc[sweep["cost_$"].idxmin(), "threshold"])
print(sweep.to_string(index=False))
print(f"\nchosen threshold (min expected cost): {CHURN_THRESHOLD}")

In [ ]:
# What drives churn? Coefficients of the (uncalibrated) logistic model.
feature_names = leader.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(leader.named_steps["clf"].coef_[0], index=feature_names)
coefs.index = [n.split("__")[1] for n in coefs.index]
top = pd.concat([coefs.nlargest(6), coefs.nsmallest(6)]).sort_values()
top.plot.barh(figsize=(7, 5),
              color=["seagreen" if v < 0 else "indianred" for v in top])
plt.title("Churn drivers (red push toward churn, green protect)")
plt.xlabel("logistic coefficient")
plt.tight_layout()
plt.show()

**Findings:** calibration fixes the inflated probabilities (Brier improves) while
ROC-AUC is unchanged — sigmoid calibration is monotone, so the ranking survives. The
cost-optimal threshold lands around ~0.3, *not* 0.5: with a 10:1 cost ratio it's worth
contacting more customers to miss fewer churners. Drivers match business intuition:
fiber-optic internet and month-to-month contracts push churn; long tenure and two-year
contracts protect.

## Step 10 — Tune the leader

Small honest grid over the regularization strength `C`, scored by 5-fold ROC-AUC.
(Phase 1 lesson: at prototype scale, expect tuning to matter less than data.)

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    Pipeline([("pre", make_preprocessor()),
              ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    param_grid={"clf__C": [0.01, 0.1, 1.0, 10.0]},
    scoring="roc_auc", cv=5, n_jobs=-1,
)
grid.fit(X_train, y_train)
print("best params:", grid.best_params_, "| CV ROC-AUC:", round(grid.best_score_, 3))

tuned_calibrated = CalibratedClassifierCV(grid.best_estimator_, method="sigmoid", cv=5)
tuned_calibrated.fit(X_train, y_train)
print("test ROC-AUC (tuned+calibrated):",
      round(roc_auc_score(y_test, tuned_calibrated.predict_proba(X_test)[:, 1]), 3))

## Step 11 — Save & predict

Refit the winning recipe on **all** data and save the model **together with its decision
threshold** — in production the model and its decision policy version together, so the
API can never load one without the other.

In [ ]:
import joblib

final_model = CalibratedClassifierCV(
    Pipeline([("pre", make_preprocessor()),
              ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                         C=grid.best_params_["clf__C"]))]),
    method="sigmoid", cv=5,
)
final_model.fit(X, y)

MODEL_PATH = Path("../models/churn_model.joblib")
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model, "threshold": CHURN_THRESHOLD}, MODEL_PATH)
print("saved", MODEL_PATH, f"(threshold={CHURN_THRESHOLD})")

In [ ]:
# Reload from disk (prove the artifact works) and score two contrasting customers.
bundle = joblib.load(MODEL_PATH)
model, threshold = bundle["model"], bundle["threshold"]

high_risk = pd.DataFrame([{
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 95.0, "TotalCharges": 190.0,
}])
low_risk = pd.DataFrame([{
    "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes",
    "tenure": 68, "PhoneService": "Yes", "MultipleLines": "Yes",
    "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
    "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Two year", "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)", "MonthlyCharges": 60.0,
    "TotalCharges": 4080.0,
}])

for label, customer in [("high-risk profile", high_risk), ("low-risk profile", low_risk)]:
    p = float(model.predict_proba(customer)[0, 1])
    print(f"{label}: churn probability={p:.1%}  will_churn={p >= threshold}")

## Overall outcome

- **Winner:** tuned + sigmoid-calibrated Logistic Regression (`class_weight="balanced"`).
- **Honest headline:** ROC-AUC ≈ 0.84 on a 1,409-customer held-out set — competitive with
  published Telco baselines; probabilities are calibrated, and the decision threshold
  (~0.3) is a documented business choice, not a default.
- **Three production lessons this phase adds:**
  1. `class_weight="balanced"` helps ranking but *breaks* probability honesty — calibrate
     before ever showing a percentage to a human.
  2. The 0.5 threshold is a convention, not a decision — pick it from costs.
  3. An explainable model (coefficients) answers the stakeholder question "why is this
     customer flagged?" for free.

Next: refactor into `src/churn.py`, expose `POST /predict-churn`, add the frontend form.